In [15]:
!pip install -U 'tensorflow[and-cuda]'
!pip install -q jiwer

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from jiwer import wer

In [16]:
print("1. Налаштування середовища")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f"Використовується GPU: {gpus[0].name}")
    except RuntimeError as e:
        print(e)
else:
    print("GPU не підключено, використовується CPU")

BATCH_SIZE = 32
EPOCHS = 50
EARLY_STOP_PATIENCE = 10
MODEL_SAVE_PATH = "/kaggle/working/deepspeech_model.keras"
FRAME_STEP = 256  
FFT_LENGTH = 256
FRAME_LENGTH = 256

1. Налаштування середовища
Використовується GPU: /physical_device:GPU:0


In [3]:
print("\n2. Підготовка даних")

ljspeech_ds = tfds.load("ljspeech", split="train", as_supervised=False)

characters = [x for x in "abcdefghijklmnopqrstuvwxyz'?! "]
char_to_num = layers.StringLookup(vocabulary=characters, oov_token="")
num_to_char = layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True)

print(f"Словник створено. Розмір: {char_to_num.vocabulary_size()}")

def encode_single_sample(sample):
    audio = sample['speech']
    label = sample['text']
    
    audio = tf.cast(audio, tf.float32)
    audio = tf.reshape(audio, [-1])
    
    spectrogram = tf.signal.stft(audio, frame_length=FRAME_LENGTH, frame_step=FRAME_STEP, fft_length=FFT_LENGTH)
    spectrogram = tf.abs(spectrogram)
    spectrogram = tf.math.pow(spectrogram, 0.5)
    
    means = tf.math.reduce_mean(spectrogram, 1, keepdims=True)
    stddevs = tf.math.reduce_std(spectrogram, 1, keepdims=True)
    spectrogram = (spectrogram - means) / (stddevs + 1e-10)

    label = tf.strings.lower(label)
    label = tf.strings.unicode_split(label, input_encoding="UTF-8")
    label = char_to_num(label)

    return spectrogram, label

def get_spec_len(spec, label):
    return tf.shape(spec)[0]

train_dataset = (
    ljspeech_ds
    .map(encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .cache() 
    .shuffle(buffer_size=1000)
    .bucket_by_sequence_length(
        element_length_func=get_spec_len,
        bucket_boundaries=[200, 300, 400, 500, 600, 700, 800],
        bucket_batch_sizes=[BATCH_SIZE] * 8,
        pad_to_bucket_boundary=False
    )
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

val_dataset = train_dataset.take(10)
train_dataset = train_dataset.skip(10)

print("Дані підготовлено.")


2. Підготовка даних


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/ljspeech/incomplete.4AY79P_1.1.1/ljspeech-train.tfrecord*...:   0%|       …

I0000 00:00:1763842725.712462      48 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


Dataset ljspeech downloaded and prepared to /root/tensorflow_datasets/ljspeech/1.1.1. Subsequent calls will reuse this data.
Словник створено. Розмір: 31
Дані підготовлено.


In [17]:
print("\n3. Архітектура моделі")

def CTCLoss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")

    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")

    loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
    return loss

def build_model(input_dim, output_dim, rnn_layers=2, rnn_units=128):
    input_spectrogram = layers.Input((None, input_dim), name="input")
    
    x = layers.Reshape((-1, input_dim, 1), name="expand_dim")(input_spectrogram)
    
    x = layers.Conv2D(32, kernel_size=[11, 41], strides=[2, 2], padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, kernel_size=[11, 21], strides=[1, 2], padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    
    new_shape = (-1, x.shape[-2] * x.shape[-1]) 
    x = layers.Reshape(target_shape=new_shape, name="reshape_rnn")(x)
    x = layers.Dense(rnn_units, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    
    for i in range(rnn_layers):
        recurrent = layers.Bidirectional(
            layers.LSTM(rnn_units, return_sequences=True), name=f"bi_lstm_{i+1}"
        )(x)
        x = layers.Dropout(0.2)(recurrent)
        
    output = layers.Dense(output_dim + 1, activation="softmax", name="output")(x)
    
    model = keras.Model(inputs=input_spectrogram, outputs=output, name="DeepSpeech_Lite")
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4), loss=CTCLoss)
    
    return model

input_dim = FFT_LENGTH // 2 + 1
model = build_model(input_dim=input_dim, output_dim=char_to_num.vocabulary_size(), rnn_units=256)
model.summary()


3. Архітектура моделі


Model: "DeepSpeech_Lite"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, None, 129)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ expand_dim (Reshape)            │ (None, None, 129, 1)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, None, 65, 32)   │        14,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, None, 65, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, None, 33, 32)   │       236,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, None, 33, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_rnn (Reshape)           │ (None, None, 1056)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, None, 256)      │       270,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_lstm_1 (Bidirectional)       │ (None, None, 512)      │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, None, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_lstm_2 (Bidirectional)       │ (None, None, 512)      │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, None, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, None, 32)       │        16,416 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,163,840 (12.07 MB)

 Trainable params: 3,163,712 (12.07 MB)

 Non-trainable params: 128 (512.00 B)

In [18]:
print("4. Навчання моделі")

early_stopper = keras.callbacks.EarlyStopping(
    monitor="val_loss", 
    patience=EARLY_STOP_PATIENCE, 
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopper],
    verbose=1 
)

model.save(MODEL_SAVE_PATH)
print(f"Модель збережено у {MODEL_SAVE_PATH}")

4. Навчання моделі
Epoch 1/50
403/403 ━━━━━━━━━━━━━━━━━━━━ 138s 329ms/step - loss: 311.9748 - val_loss: 333.9516
Epoch 2/50
403/403 ━━━━━━━━━━━━━━━━━━━━ 133s 329ms/step - loss: 296.5890 - val_loss: 313.7284
Epoch 3/50
403/403 ━━━━━━━━━━━━━━━━━━━━ 133s 331ms/step - loss: 291.7042 - val_loss: 298.6271
Epoch 4/50
403/403 ━━━━━━━━━━━━━━━━━━━━ 133s 330ms/step - loss: 282.0326 - val_loss: 275.9454
Epoch 5/50
403/403 ━━━━━━━━━━━━━━━━━━━━ 131s 325ms/step - loss: 258.7904 - val_loss: 274.1889
Epoch 6/50
403/403 ━━━━━━━━━━━━━━━━━━━━ 136s 336ms/step - loss: 233.4528 - val_loss: 258.6005
Epoch 7/50
403/403 ━━━━━━━━━━━━━━━━━━━━ 131s 325ms/step - loss: 212.7980 - val_loss: 224.1329
Epoch 8/50
403/403 ━━━━━━━━━━━━━━━━━━━━ 131s 324ms/step - loss: 198.4129 - val_loss: 202.2898
Epoch 9/50
403/403 ━━━━━━━━━━━━━━━━━━━━ 131s 325ms/step - loss: 186.1628 - val_loss: 202.9449
Epoch 10/50
403/403 ━━━━━━━━━━━━━━━━━━━━ 137s 339ms/step - loss: 177.6207 - val_loss: 197.6822
Epoch 11/50
403/403 ━━━━━━━━━━━━━━━━━━━━

In [22]:
print("\n5. Тестування моделі")

def decode_batch_predictions(pred):
    input_len = np.ones(pred.shape[0]) * pred.shape[1]
    results = keras.backend.ctc_decode(pred, input_length=input_len, greedy=True)[0][0]
    
    output_text = []
    for result in results:
        text = tf.strings.reduce_join(num_to_char(result)).numpy().decode("utf-8")
        output_text.append(text)
    return output_text

for batch in val_dataset.take(1):
    spectrograms = batch[0]
    labels = batch[1]

    preds = model.predict(spectrograms, verbose=0)
    pred_texts = decode_batch_predictions(preds)

    n_examples = min(10, len(pred_texts))
    total_wer = 0

    print("\n===== РЕЗУЛЬТАТИ ТЕСТУВАННЯ =====")
    print(f"Кількість прикладів у batch: {len(pred_texts)}\n")

    for i in range(n_examples):
        print(f"Приклад №{i + 1}")
        print("-" * 70)

        true_label = tf.strings.reduce_join(num_to_char(labels[i])).numpy().decode("utf-8")
        pred_label = pred_texts[i]

        example_wer = wer(true_label, pred_label)
        total_wer += example_wer

        print(f"Справжній текст:\n   {true_label}")
        print(f"Передбачений текст:\n   {pred_label}")
        print(f"Word Error Rate (WER): {example_wer:.3f}")
        print(f"Розмір спектрограми: {spectrograms[i].shape}\n")

    avg_wer = total_wer / n_examples
    print("=" * 70)
    print(f"Середній WER по {n_examples} прикладах: {avg_wer:.3f}")
    print("=" * 70)


5. Тестування моделі

===== РЕЗУЛЬТАТИ ТЕСТУВАННЯ =====
Кількість прикладів у batch: 32

Приклад №1
----------------------------------------------------------------------
Справжній текст:
   after a review of these inconsistencies in his testimony before the commission whaley was interviewed again in dallas
Передбачений текст:
   after a revewve bese in conistencies ind his testimony before the commission weilly was interviewdygan in da
Word Error Rate (WER): 0.500
Розмір спектрограми: (699, 129)

Приклад №2
----------------------------------------------------------------------
Справжній текст:
   the jail authorities often gave the highest possibly undue importance to the value of remunerative employment
Передбачений текст:
   the jail ofthoritis oftingae the hies osibly ondu importans to the va ofve ereueartve
Word Error Rate (WER): 0.688
Розмір спектрограми: (699, 129)

Приклад №3
----------------------------------------------------------------------
Справжній текст:
   it is now e